# Pharmasense — TFT Training (Complete Drug Demand)
**Temporal Fusion Transformer — full drug supply chain demand forecasting**

## What this model covers
The TFT is the **demand baseline engine** for the ENTIRE drug supply chain — both communicable and non-communicable:

| Disease / Condition | Type | Data Source | Why TFT not ST-GNN |
|---|---|---|---|
| `acute_diarrhoeal_disease` | Communicable (endemic) | EpiClim IDSP | Seasonal/waterborne — no spatial spread |
| `dengue` | Communicable (vector) | EpiClim IDSP | TFT handles the *steady-state baseline*; ST-GNN handles outbreaks |
| `malaria` | Communicable (vector) | EpiClim IDSP | Same — baseline here, outbreak in ST-GNN |
| `cholera` | Communicable (waterborne) | EpiClim IDSP | Same |
| `hypertension` | NCD / Chronic | **NFHS-5 MBQ** | Pure time-series, no adjacency |
| `type2_diabetes` | NCD / Chronic | **NFHS-5 MBQ** | Pure time-series |
| `tuberculosis` | Infectious (DOTS) | **RNTCP 2022 MBQ** | Predictable fixed-course demand |
| `anaemia` | NCD / Chronic | **NFHS-5 MBQ** | Pure time-series |

**Total training series:** ~5,000+ (district × condition) time series from 2018–2024

**Output:** `tft_best.ckpt` + `tft_latest.pt` — 7-quantile forecasts per series per week

## Before running
1. **Runtime → Change runtime type → T4 GPU**
2. **Add Colab Secret:** 🔑 sidebar → `PHARMASENSE_DB_PASSWORD`
3. **Runtime → Run All**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhi-dev99/agtechathon/blob/main/ml/notebooks/02_train_tft.ipynb)

In [ ]:
import subprocess, sys
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                   capture_output=True, text=True)
if r.returncode == 0:
    print(f'GPU: {r.stdout.strip()}')
else:
    raise RuntimeError('No GPU. Runtime → Change runtime type → T4 GPU')

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'pytorch-forecasting==1.0.0', 'pytorch-lightning==2.2.4',
    'psycopg2-binary', 'pandas', 'numpy', 'scikit-learn', 'matplotlib'], check=True)
print('Dependencies installed.')

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────────
from google.colab import userdata

DB_CONFIG = dict(
    host='aws-1-ap-south-1.pooler.supabase.com', port=6543,
    dbname='postgres', user='postgres.xwtvujwdybvjbaucshbt',
    password=userdata.get('PHARMASENSE_DB_PASSWORD'), sslmode='require',
)

# Communicable diseases from epid_weekly (real outbreak records)
COMM_DISEASES = ['acute_diarrhoeal_disease', 'dengue', 'malaria', 'cholera']
# NCDs from ncd_demand_weekly (NFHS-5 morbidity-based quantification)
NCD_TYPES     = ['hypertension', 'type2_diabetes', 'tuberculosis', 'anaemia']

# Hyperparameters
MAX_ENCODER_LEN     = 24    # 6 months lookback
MAX_PREDICTION_LEN  = 4     # 4-week (1 month) horizon
MIN_OBS_PER_GROUP   = 30    # min weeks per series to include in training
BATCH_SIZE          = 128   # larger batch — bigger dataset now
MAX_EPOCHS          = 60
LEARNING_RATE       = 3e-3
HIDDEN_SIZE         = 48    # slightly larger — more series types to learn from
ATTENTION_HEADS     = 4
DROPOUT             = 0.1
HIDDEN_CONT_SIZE    = 20

DRIVE_OUTPUT = '/content/drive/MyDrive/pharmasense_models/'

import torch
print(f'CUDA: {torch.cuda.is_available()}')
print(f'Communicable diseases: {COMM_DISEASES}')
print(f'NCDs: {NCD_TYPES}')
print(f'Total condition types: {len(COMM_DISEASES) + len(NCD_TYPES)}')

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f'Drive ready: {DRIVE_OUTPUT}')

In [ ]:
# ── Load ALL data — communicable + NCD ────────────────────────────────────────
import psycopg2
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

conn = psycopg2.connect(**DB_CONFIG)

# District static features (centroid, state, population)
dist_df = pd.read_sql("""
    SELECT id, name, state, centroid_lat, centroid_lon,
           COALESCE(population, 50000) AS population
    FROM districts;
""", conn)

# ── Source 1: Communicable disease data (real IDSP outbreak records) ──────────
comm_df = pd.read_sql("""
    SELECT district_id,
           disease          AS condition_type,
           week_start_date,
           cases            AS raw_value,
           'communicable'   AS data_family
    FROM epid_weekly
    WHERE disease IN ('acute_diarrhoeal_disease','dengue','malaria','cholera')
    ORDER BY week_start_date, district_id;
""", conn)

# ── Source 2: NCD demand data (NFHS-5 morbidity-based quantification) ─────────
ncd_df = pd.read_sql("""
    SELECT district_id,
           ncd_type         AS condition_type,
           week_start_date,
           patients_est     AS raw_value,
           'ncd'            AS data_family
    FROM ncd_demand_weekly
    ORDER BY week_start_date, district_id;
""", conn)
conn.close()

print(f'Communicable records: {len(comm_df):,}')
print(f'NCD records:          {len(ncd_df):,}')

# ── Combine into unified dataframe ────────────────────────────────────────────
df = pd.concat([comm_df, ncd_df], ignore_index=True)
df['week_start_date'] = pd.to_datetime(df['week_start_date'])
df['log_value']       = np.log1p(df['raw_value'].clip(lower=0).astype(float))

# Merge district static features
df = df.merge(dist_df[['id','state','centroid_lat','centroid_lon','population']],
              left_on='district_id', right_on='id', how='left').drop('id', axis=1)
df['log_population'] = np.log1p(df['population'].fillna(50000).astype(float))

print(f'\nCombined dataset: {len(df):,} records')
print(f'Condition types:  {df["condition_type"].nunique()}')
print(f'Districts:        {df["district_id"].nunique()}')
print(f'Date range:       {df["week_start_date"].min().date()} -> {df["week_start_date"].max().date()}')
print()
print(df.groupby(['data_family','condition_type'])['raw_value'].agg(['count','mean']).round(1))

In [ ]:
# ── Feature engineering ────────────────────────────────────────────────────────

# Categorical encoders
le_district   = LabelEncoder()
le_state      = LabelEncoder()
le_condition  = LabelEncoder()
le_family     = LabelEncoder()

df['district_enc']  = le_district.fit_transform(df['district_id'].astype(str))
df['state_enc']     = le_state.fit_transform(df['state'].fillna('Unknown').astype(str))
df['condition_enc'] = le_condition.fit_transform(df['condition_type'].astype(str))
df['family_enc']    = le_family.fit_transform(df['data_family'].astype(str))

# Integer time index from the global minimum date
min_date        = df['week_start_date'].min()
df['time_idx']  = ((df['week_start_date'] - min_date).dt.days // 7).astype(int)

# Known future time-varying covariates (calendar features)
df['month']     = df['week_start_date'].dt.month.astype(float)   # seasonality signal
df['week_of_year'] = df['week_start_date'].dt.isocalendar().week.astype(float)
df['year_frac'] = (df['week_start_date'].dt.year - 2018) + df['month'] / 12  # trend proxy

# Unique group per (district, condition) — one independent time series each
df['group_id']  = df['district_id'].astype(str) + '__' + df['condition_type']

# Filter: only keep groups with enough history
gc           = df.groupby('group_id')['time_idx'].count()
valid        = gc[gc >= MIN_OBS_PER_GROUP].index
df           = df[df['group_id'].isin(valid)].copy()

print(f'Records after min-obs filter ({MIN_OBS_PER_GROUP} weeks): {len(df):,}')
print(f'Valid (district x condition) series: {df["group_id"].nunique():,}')
by_condition = df.groupby('condition_type')['group_id'].nunique().sort_values(ascending=False)
print(f'\nSeries per condition type:')
print(by_condition.to_string())

In [ ]:
# ── Build TimeSeriesDataSet ────────────────────────────────────────────────────
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss
from torch.utils.data import DataLoader

# Chronological train/val split — 80% train, 20% val
max_t      = df['time_idx'].max()
cutoff     = int(max_t * 0.80)
train_df   = df[df['time_idx'] <= cutoff]

training = TimeSeriesDataSet(
    train_df,
    time_idx                  = 'time_idx',
    target                    = 'log_value',
    group_ids                 = ['group_id'],
    max_encoder_length        = MAX_ENCODER_LEN,
    max_prediction_length     = MAX_PREDICTION_LEN,

    # Static: don't change per series — location, state, condition type
    static_categoricals       = ['district_enc', 'state_enc', 'condition_enc', 'family_enc'],
    static_reals              = ['centroid_lat', 'centroid_lon', 'log_population'],

    # Known future: calendar features we always know in advance
    time_varying_known_reals  = ['time_idx', 'month', 'week_of_year', 'year_frac'],

    # Unknown future: the target (and any observed-but-not-predictable features)
    time_varying_unknown_reals = ['log_value'],

    # GroupNormalizer: learns separate mean/std per (district, condition) series
    # Critical for combining NCD (patients in thousands) with epidemic data (cases 0-10000)
    target_normalizer         = GroupNormalizer(groups=['group_id'], transformation='log1p'),

    allow_missing_timesteps   = True,   # handles sparse epidemic data
    add_relative_time_idx     = True,   # position-awareness within encoder window
    add_target_scales         = True,   # scale-awareness per group
)

validation = TimeSeriesDataSet.from_dataset(training, df, predict=True, stop_randomization=True)

train_dl = training.to_dataloader(train=True,  batch_size=BATCH_SIZE,     num_workers=2, shuffle=True)
val_dl   = validation.to_dataloader(train=False, batch_size=BATCH_SIZE*2, num_workers=2)

print(f'Training samples:   {len(training):,}')
print(f'Validation samples: {len(validation):,}')
print(f'Batches/epoch:      {len(train_dl):,}')

In [ ]:
# ── TFT Model ──────────────────────────────────────────────────────────────────
# Key design points:
# - HIDDEN_SIZE=48: larger than before — 8 condition types require more capacity
# - GroupNormalizer handles the scale disparity between NCD patients (10K+)
#   and epidemic cases (0-5000) — each group learns its own scale
# - 7 quantile outputs: 0.02/0.1/0.25/0.5/0.75/0.9/0.98
#   API surfaces: q50=demand_baseline, q90=procurement_buffer, q02=min_stock
# - Variable selection networks: TFT learns AUTOMATICALLY which covariates
#   matter per condition (e.g. 'month' matters more for cholera than for HTN)

tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate            = LEARNING_RATE,
    hidden_size              = HIDDEN_SIZE,
    attention_head_size      = ATTENTION_HEADS,
    dropout                  = DROPOUT,
    hidden_continuous_size   = HIDDEN_CONT_SIZE,
    output_size              = 7,
    loss                     = QuantileLoss(),
    log_interval             = 20,
    reduce_on_plateau_patience = 5,
    optimizer                = 'adam',
)

total_params = sum(p.numel() for p in tft.parameters() if p.requires_grad)
print(f'TFT | params: {total_params:,} | hidden: {HIDDEN_SIZE} | heads: {ATTENTION_HEADS}')
print(f'Quantiles: {list(tft.loss.quantiles.numpy().round(2))}')
print(f'Condition types in training: {le_condition.classes_.tolist()}')

In [ ]:
# ── Train ──────────────────────────────────────────────────────────────────────
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor
from pytorch_lightning.loggers import TensorBoardLogger

callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, mode='min', verbose=True),
    ModelCheckpoint(
        dirpath='tft_best', filename='tft_checkpoint',
        monitor='val_loss', mode='min', save_top_k=1, verbose=True,
    ),
    LearningRateMonitor(logging_interval='epoch'),
]

trainer = pl.Trainer(
    max_epochs          = MAX_EPOCHS,
    accelerator         = 'gpu' if torch.cuda.is_available() else 'cpu',
    devices             = 1,
    gradient_clip_val   = 0.1,
    callbacks           = callbacks,
    enable_progress_bar = True,
    logger              = TensorBoardLogger(DRIVE_OUTPUT + 'tb_logs/', name='tft'),
)

trainer.fit(tft, train_dataloaders=train_dl, val_dataloaders=val_dl)
print(f'Best checkpoint: {callbacks[1].best_model_path}')
print(f'Best val loss:   {float(callbacks[1].best_model_score):.5f}')

In [ ]:
# ── Evaluate ───────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

best_tft = TemporalFusionTransformer.load_from_checkpoint(callbacks[1].best_model_path)
best_tft.eval()
predictions = best_tft.predict(val_dl, mode='raw', return_x=True,
                               trainer_kwargs={'accelerator': 'gpu' if torch.cuda.is_available() else 'cpu'})

# Sample forecasts — show one per condition type
n = len(validation)
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for i, ax in enumerate(axes.flat):
    idx = i * (n // 8)
    best_tft.plot_prediction(predictions.x, predictions.output, idx=min(idx, n-1),
                             ax=ax, show_future_observed=True)
    # Get group label for the plot
    group_name = df.iloc[min(idx, len(df)-1)]['group_id'].split('__')[-1] if idx < len(df) else ''
    ax.set_title(f'{group_name}', fontsize=9)
plt.suptitle('TFT Forecasts — Communicable + NCD (7-quantile)', fontsize=12)
plt.tight_layout()
plt.savefig(DRIVE_OUTPUT + 'tft_forecasts_all_conditions.png', dpi=120)
plt.show()

# Variable importance — shows which features TFT learned to use most
try:
    raw_pred, x = best_tft.predict(val_dl, mode='raw', return_x=True)
    imp = best_tft.calculate_prediction_actual_by_variable(x, raw_pred)
    best_tft.plot_prediction_actual_by_variable(imp)
    plt.savefig(DRIVE_OUTPUT + 'tft_variable_importance.png', dpi=120)
    plt.show()
except Exception as e:
    print(f'Variable importance plot skipped: {e}')

In [ ]:
# ── Save artifact ──────────────────────────────────────────────────────────────
import torch
from datetime import datetime
import shutil

ckpt_path = callbacks[1].best_model_path

# Copy checkpoint to Drive for persistence
drive_ckpt = DRIVE_OUTPUT + 'tft_best.ckpt'
shutil.copy2(ckpt_path, drive_ckpt)

artifact = {
    'model_checkpoint_path': drive_ckpt,
    'dataset_parameters':    training.get_parameters(),
    'training_config': {
        'communicable_diseases': COMM_DISEASES,
        'ncd_types':             NCD_TYPES,
        'all_conditions':        le_condition.classes_.tolist(),
        'max_encoder_length':    MAX_ENCODER_LEN,
        'max_prediction_length': MAX_PREDICTION_LEN,
        'hidden_size':           HIDDEN_SIZE,
        'attention_heads':       ATTENTION_HEADS,
        'quantiles':             list(tft.loss.quantiles.numpy()),
        'notes': {
            'q50':  'Median baseline demand — primary procurement signal',
            'q90':  'Safety stock upper bound — procurement buffer',
            'q0.1': 'Minimum expected demand — avoid overstock/wastage',
            'groupnormalizer': 'Each (district, condition) series normalized independently — scale-agnostic'
        }
    },
    'label_encoders': {
        'district':  le_district,
        'state':     le_state,
        'condition': le_condition,
        'family':    le_family,
    },
    'data_sources': {
        'communicable': 'EpiClim IDSP 2009-2022',
        'ncd':          'NFHS-5 MBQ + RNTCP 2022 (2018-2024, 641 districts)',
    },
    'best_val_loss': float(callbacks[1].best_model_score),
    'trained_at':    datetime.now().isoformat(),
    'n_series':      df['group_id'].nunique(),
    'n_records':     len(df),
}

meta_path = DRIVE_OUTPUT + 'tft_latest.pt'
torch.save(artifact, meta_path)
print(f'Artifact:   {meta_path}')
print(f'Checkpoint: {drive_ckpt}')
print(f'Conditions: {le_condition.classes_.tolist()}')

from google.colab import files
files.download(meta_path)
files.download(drive_ckpt)
print('Downloads started. Save both to ml/artifacts/ in your local repo.')